Columns explanations:
date- date of recorded price
<symbol>- closing price for recorded date for that asset
<symbol>_m20- 20 day momentum for asset
<symbol>_m50- 50 day momentum for asset
<symbol>_m100- 100 day momentum for asset
<symbol>_sd10- rolling 10 day standard deviation for asset
<symbol>_sd30- rolling 30 day standard deviation for asset
<symbol>_ewma- EWMA (exponentially weighted moving average, lambda=0.94) volatility for asset
<symbol>_garch- GARCH(1,1) volatility for asset (numbers are annualized so expect to be higher)
<symbol>_dd30- rolling 30 day downside deviation for asset"""

In [ ]:
import pandas as pd
import numpy as np
import re

# Load + prep
Futures_Data = pd.read_csv('data_testing.csv', parse_dates=['date'])
# Drop columns that are entirely empty (all NaN)
Futures_Data = Futures_Data.dropna(axis=1, how='all')
Futures_Data = Futures_Data.sort_values('date').set_index('date')

# Identify base symbols (price columns without a suffix)
base_syms = [c for c in Futures_Data.columns if '_' not in c]

# 1) Compute daily simple returns for each symbol
for sym in base_syms:
    Futures_Data[f'{sym}_ret'] = Futures_Data[sym].pct_change(fill_method=None)

# 2) Rolling mean return over 30 days (MAR = 0), annualize with sqrt(252)
ROLL = 30
ANN = np.sqrt(252)

for sym in base_syms:
    ret_col = f'{sym}_ret'
    dd_col  = f'{sym}_dd30'      # provided in your file
    sort_col = f'{sym}_sortino30'

    # rolling average of daily returns (0 MAR)
    roll_mean = Futures_Data[ret_col].rolling(ROLL, min_periods=ROLL).mean()

    # Sortino: (mean excess return) / downside deviation
    Futures_Data[sort_col] = (roll_mean / Futures_Data[dd_col]) * ANN

# Optional: overall (full-sample) Sortino per symbol from raw returns
def sortino(series, mar=0.0):
    r = series.dropna()
    if r.empty:
        return np.nan
    downside = r[r < mar]
    dd = downside.sub(mar).pow(2).mean() ** 0.5
    if dd == 0 or np.isnan(dd):
        return np.nan
    # annualize: daily mean * 252 divided by daily downside * sqrt(252)
    return (r.mean() * 252) / (dd * np.sqrt(252))

overall_sortino = {
    sym: sortino(Futures_Data[f'{sym}_ret'])
    for sym in base_syms
}

print("Columns added:", [f"{sym}_sortino30" for sym in base_syms])
print("Overall Sortino (full sample):", overall_sortino)

Columns added: ['6B_sortino30', '6C_sortino30', '6E_sortino30', '6J_sortino30', '6M_sortino30', '6N_sortino30', '6S_sortino30', 'CL_sortino30', 'ES_sortino30', 'GC_sortino30', 'GF_sortino30', 'HE_sortino30', 'HG_sortino30', 'KE_sortino30', 'LE_sortino30', 'NQ_sortino30', 'PL_sortino30', 'RB_sortino30', 'RTY_sortino30', 'SI_sortino30', 'UB_sortino30', 'YM_sortino30', 'ZC_sortino30', 'ZL_sortino30', 'ZM_sortino30', 'ZN_sortino30', 'ZR_sortino30', 'ZS_sortino30', 'ZW_sortino30']
Overall Sortino (full sample): {'6B': np.float64(0.6184591194572312), '6C': np.float64(-0.17539007806499957), '6E': np.float64(0.6212280304994019), '6J': np.float64(-0.26014685255951986), '6M': np.float64(-0.060187511366301204), '6N': np.float64(0.14411328187020664), '6S': np.float64(0.39972184203020816), 'CL': np.float64(0.17103446012033946), 'ES': np.float64(1.7635745633459907), 'GC': np.float64(0.46671263523842627), 'GF': np.float64(1.8064027430850644), 'HE': np.float64(0.6107565789405738), 'HG': np.float64(0.2

In [3]:
import pandas as pd
import numpy as np

# Simple backtest: each rebalance date, pick top-K by rolling Sortino and hold 1/N
K = 5
REB_MONTHS = 1  # rebalance monthly
ROLL = 30

# Use symbols we have returns for
symbols = [c[:-4] for c in Futures_Data.columns if c.endswith('_ret')]

# Compute a rolling Sortino on returns using provided dd30 series
ANN = np.sqrt(252)
for sym in symbols:
    ret = Futures_Data[f'{sym}_ret']
    dd  = Futures_Data.get(f'{sym}_dd30')
    if dd is None:
        continue
    roll_mean = ret.rolling(ROLL, min_periods=ROLL).mean()
    Futures_Data[f'{sym}_sortino_roll'] = (roll_mean / dd) * ANN

# Monthly rebalancing dates (end of month)
rebalance_dates = Futures_Data.resample('M').last().index

# Backtest loop
portfolio_nav = pd.Series(index=Futures_Data.index, dtype=float)
portfolio_nav.iloc[0] = 1.0
weights = pd.DataFrame(0.0, index=Futures_Data.index, columns=symbols)
current_weights = pd.Series(0.0, index=symbols)

for dt in Futures_Data.index:
    # Rebalance on scheduled dates (at open of next day, approximate by same day close-to-close)
    if dt in rebalance_dates:
        # rank by latest rolling sortino, drop missing
        latest_scores = pd.Series({s: Futures_Data.at[dt, f'{s}_sortino_roll'] if f'{s}_sortino_roll' in Futures_Data.columns else np.nan for s in symbols})
        top = latest_scores.dropna().sort_values(ascending=False).head(K).index
        if len(top) > 0:
            current_weights = pd.Series(0.0, index=symbols)
            current_weights.loc[top] = 1.0 / len(top)
        else:
            current_weights = pd.Series(0.0, index=symbols)
    
    weights.loc[dt] = current_weights
    # Compute portfolio return for the day
    daily_rets = pd.Series({s: Futures_Data.at[dt, f'{s}_ret'] if f'{s}_ret' in Futures_Data.columns else 0.0 for s in symbols})
    port_ret = np.nansum(current_weights.values * daily_rets.fillna(0.0).values)
    if pd.isna(portfolio_nav.loc[dt]):
        # carry from previous NAV
        prev = portfolio_nav.loc[:dt].iloc[-2] if portfolio_nav.loc[:dt].size > 1 else 1.0
        portfolio_nav.loc[dt] = prev * (1.0 + port_ret)
    else:
        portfolio_nav.loc[dt] *= (1.0 + port_ret)

# Basic performance stats
nav = portfolio_nav.dropna()
port_rets = nav.pct_change().dropna()
sharpe = np.sqrt(252) * port_rets.mean() / port_rets.std() if port_rets.std() not in (0, np.nan) else np.nan

print("Backtest complete. Final NAV:", float(nav.iloc[-1]))
print("Annualized Sharpe (simple):", float(sharpe) if pd.notna(sharpe) else np.nan)

# Optional: display last few weights and NAV
display(weights.asfreq('M').tail())
display(nav.tail())


/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/79265282.py:23: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rebalance_dates = Futures_Data.resample('M').last().index


Backtest complete. Final NAV: 1.9226243911702317
Annualized Sharpe (simple): 0.33376806630607697


/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/79265282.py:63: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  display(weights.asfreq('M').tail())


,6B,6C,6E,6J,6M,6N,6S,CL,ES,GC,...,SI,UB,YM,ZC,ZL,ZM,ZN,ZR,ZS,ZW
date,,,,,,,,,,,,,,,,,,,,,
2025-03-31,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.20,0.0,0.0,...,0.0,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-04-30,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,...,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-06-30,0.0,0.00,0.2,0.0,0.0,0.0,0.2,0.00,0.2,0.0,...,0.0,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-07-31,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.20,0.2,0.0,...,0.0,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0


date
2025-08-21    1.911547
2025-08-22    1.931537
2025-08-24    1.930323
2025-08-25    1.932368
2025-08-26    1.922624
dtype: float64

In [ ]:
import pandas as pd
import numpy as np

# Volatility-Weighted Momentum backtest with percentile triggers
# - score = momentum / volatility
# - enter long when score >= rolling PCTL (e.g., 90th) with 1-year warmup
# - no look-ahead: use only info up to t-1 for trading at t
# - exposure caps: per-asset and gross

TRADING_DAYS_PER_YEAR = 252

def compute_score(df: pd.DataFrame, symbol: str, mom_col: str, vol_col: str) -> pd.Series:
    mcol = f"{symbol}_{mom_col}" if not mom_col.startswith(symbol) else mom_col
    vcol = f"{symbol}_{vol_col}" if not vol_col.startswith(symbol) else vol_col
    if mcol not in df.columns or vcol not in df.columns:
        return pd.Series(index=df.index, dtype=float)

    score = df[mcol] / df[vcol] ### score is momentum / volatility
    return score.replace([np.inf, -np.inf], np.nan)

def rolling_percentile(series: pd.Series, window: int, pctl: float) -> pd.Series:
    # rolling percentile with min_periods=window to enforce warmup
    
    return series.rolling(window=window, min_periods=window).quantile(pctl / 100.0)

def backtest_vwm(
    df: pd.DataFrame,
    momentum_metric: str = 'm50',
    vol_metric: str = 'dd30',
    entry_percentile: float = 90.0,
    warmup_days: int = TRADING_DAYS_PER_YEAR,
    rebalance_freq: str = 'D',  # 'D' daily or 'M' monthly
    per_asset_cap: float = 0.1,
    gross_exposure_cap: float = 1.0,
):
    # Symbols
    symbols = [c for c in df.columns if ('_' not in c) and (c != 'date')]

    # Build score table (no look-ahead applied later via shift)
    score_cols = {}
    for sym in symbols:
        s = compute_score(df, sym, momentum_metric, vol_metric)
        if s.notna().any():
            sc_name = f"{sym}_score_{momentum_metric}_{vol_metric}"
            df[sc_name] = s
            score_cols[sym] = sc_name

    if not score_cols:
        raise ValueError("No score columns found. Check metric names against available columns.")

    # Rolling percentile thresholds and look-ahead safe signals
    thresholds = {}
    signals = {}
    for sym, sc_name in score_cols.items():
        thr = rolling_percentile(df[sc_name], warmup_days, entry_percentile)
        thresholds[sym] = thr
        # Enter long when score >= threshold. Use t-1 info for trading at t.
        sig = (df[sc_name] >= thr).shift(1).fillna(False)
        signals[sym] = sig.astype(float)  # 1.0 in market, 0.0 out

    # Rebalance calendar
    if rebalance_freq == 'M':
        rebalance_dates = df.resample('M').last().index
    else:
        rebalance_dates = df.index  # daily

    # Initialize
    nav = pd.Series(index=df.index, dtype=float)
    nav.iloc[0] = 1.0
    weights = pd.DataFrame(0.0, index=df.index, columns=symbols)
    current_w = pd.Series(0.0, index=symbols)

    # Iterate
    for dt in df.index:
        # Determine target weights only on rebalance dates
        if dt in rebalance_dates:
            raw_sig = pd.Series({sym: signals[sym].loc[dt] if sym in signals else 0.0 for sym in symbols})
            # Apply per-asset cap and scale to gross cap
            if raw_sig.sum() > 0:
                tgt = raw_sig / raw_sig.sum()  # equal weight among active
            else:
                tgt = pd.Series(0.0, index=symbols)
            tgt = tgt.clip(upper=per_asset_cap)
            total = tgt.sum()
            if total > 0:
                scale = min(gross_exposure_cap / total, 1.0)
                tgt = tgt * scale
            current_w = tgt.reindex(symbols).fillna(0.0)

        weights.loc[dt] = current_w
        # Compute daily portfolio return
        daily_rets = pd.Series({sym: df.at[dt, f"{sym}_ret"] if f"{sym}_ret" in df.columns else 0.0 for sym in symbols})
        port_ret = float(np.nansum(current_w.values * daily_rets.fillna(0.0).values))
        # NAV update
        if pd.isna(nav.loc[dt]):
            prev = nav.loc[:dt].iloc[-2] if nav.loc[:dt].size > 1 else 1.0
            nav.loc[dt] = prev * (1.0 + port_ret)
        else:
            nav.loc[dt] *= (1.0 + port_ret)

    # Performance
    nav = nav.dropna()
    rets = nav.pct_change().dropna()
    ann_ret = (1 + rets.mean()) ** TRADING_DAYS_PER_YEAR - 1 if not rets.empty else np.nan
    ann_vol = rets.std() * np.sqrt(TRADING_DAYS_PER_YEAR) if not rets.empty else np.nan
    sharpe = ann_ret / ann_vol if (ann_vol not in (0, np.nan) and pd.notna(ann_vol)) else np.nan
    max_dd = (nav / nav.cummax() - 1).min() if not nav.empty else np.nan

    stats = {
        'momentum': momentum_metric,
        'vol': vol_metric,
        'entry_pctl': entry_percentile,
        'rebalance': rebalance_freq,
        'per_asset_cap': per_asset_cap,
        'gross_cap': gross_exposure_cap,
        'final_nav': float(nav.iloc[-1]) if not nav.empty else np.nan,
        'ann_return': float(ann_ret) if pd.notna(ann_ret) else np.nan,
        'ann_vol': float(ann_vol) if pd.notna(ann_vol) else np.nan,
        'sharpe': float(sharpe) if pd.notna(sharpe) else np.nan,
        'max_drawdown': float(max_dd) if pd.notna(max_dd) else np.nan,
    }

    return nav, weights, pd.DataFrame([stats])

# Example single run (edit as needed)
example_nav, example_w, example_stats = backtest_vwm(
    Futures_Data,
    momentum_metric='m50',
    vol_metric='dd30',
    entry_percentile=90.0,
    warmup_days=TRADING_DAYS_PER_YEAR,
    rebalance_freq='D',
    per_asset_cap=0.1,
    gross_exposure_cap=1.0,
)

print("Example VWM stats:")
display(example_stats)
display(example_nav.tail())
display(example_w.asfreq('M').tail())


/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[sc_name] = s
/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[sc_name] = s
/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

Example VWM stats:


,momentum,vol,entry_pctl,rebalance,per_asset_cap,gross_cap,final_nav,ann_return,ann_vol,sharpe,max_drawdown
0,m50,dd30,90.0,D,0.1,1.0,1.15887,0.008451,0.033041,0.255766,-0.120595


date
2025-08-21    1.156881
2025-08-22    1.157093
2025-08-24    1.157122
2025-08-25    1.157122
2025-08-26    1.158870
dtype: float64

/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:138: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  display(example_w.asfreq('M').tail())


,6B,6C,6E,6J,6M,6N,6S,CL,ES,GC,...,SI,UB,YM,ZC,ZL,ZM,ZN,ZR,ZS,ZW
date,,,,,,,,,,,,,,,,,,,,,
2025-03-31,0.1,0.0,0.1,0.0,0.1,0.1,0.1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-04-30,0.0,0.1,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.1,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
2025-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,...,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.1,0.0,0.0
2025-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0


In [ ]:
import itertools

# Grid run across available momentum and vol columns present in the data
mom_options = ['m20', 'm50', 'm100']
vol_options = ['sd10', 'sd20', 'ewma', 'garch', 'dd30'] # 

# Filter to combos that exist for at least one symbol
symbols = [c for c in Futures_Data.columns if ('_' not in c) and (c != 'date')]

available = []
for m, v in itertools.product(mom_options, vol_options):
    exists_any = any((f"{s}_{m}" in Futures_Data.columns) and (f"{s}_{v}" in Futures_Data.columns) for s in symbols)
    if exists_any:
        available.append((m, v))

results = []
navs = {}
for m, v in available:
    nav, w, stats = backtest_vwm(
        Futures_Data,
        momentum_metric=m, vol_metric=v,
        entry_percentile=90.0,
        warmup_days=TRADING_DAYS_PER_YEAR,
        rebalance_freq='D',
        per_asset_cap=0.1,
        gross_exposure_cap=1.0,
    )
    results.append(stats)
    navs[(m, v)] = nav

summary = pd.concat(results, ignore_index=True).sort_values('sharpe', ascending=False)
print("VWM grid summary (top 10 by Sharpe):")
display(summary.head(10))



/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[sc_name] = s
/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[sc_name] = s
/var/folders/dm/f8k7qw0s21728mv51zsnqr5h0000gn/T/ipykernel_11019/1227582339.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

VWM grid summary (top 10 by Sharpe):


,momentum,vol,entry_pctl,rebalance,per_asset_cap,gross_cap,final_nav,ann_return,ann_vol,sharpe,max_drawdown
6,m50,sd20,90.0,D,0.1,1.0,1.211701,0.010782,0.030865,0.349325,-0.082393
9,m50,dd30,90.0,D,0.1,1.0,1.158870,0.008451,0.033041,0.255766,-0.120595
5,m50,sd10,90.0,D,0.1,1.0,1.114694,0.006264,0.030013,0.208723,-0.084684
7,m50,ewma,90.0,D,0.1,1.0,1.077733,0.004527,0.032277,0.140249,-0.113407
14,m100,dd30,90.0,D,0.1,1.0,1.033663,0.002338,0.033723,0.069344,-0.112770
4,m20,dd30,90.0,D,0.1,1.0,1.025164,0.001823,0.031425,0.058002,-0.171419
10,m100,sd10,90.0,D,0.1,1.0,1.000582,0.000492,0.030378,0.016206,-0.111941
13,m100,garch,90.0,D,0.1,1.0,0.998618,-0.000003,0.011939,-0.000212,-0.046391
2,m20,ewma,90.0,D,0.1,1.0,0.958451,-0.001804,0.030268,-0.059615,-0.198084
12,m100,ewma,90.0,D,0.1,1.0,0.923260,-0.003703,0.033210,-0.111499,-0.127965
